# ex08 · softmax回归的简洁实现（对应教材 3.7）

> **做题流程**：补全 TODO，运行自测；做完再看 `solutions/ex08-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 本节用 nn 模块重写 ex07，关键区别：**nn.CrossEntropyLoss 已经内置 softmax**，所以模型里不要再写 softmax。

In [1]:
import torch
import torchvision
from torch.utils import data
from torchvision import transforms

def load_data_fashion_mnist(batch_size):
    trans = transforms.ToTensor()
    mnist_train = torchvision.datasets.FashionMNIST(
        root='../data', train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root='../data', train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True),
            data.DataLoader(mnist_test, batch_size, shuffle=False))

batch_size = 256
train_iter, test_iter = load_data_fashion_mnist(batch_size)
print('训练集批数:', len(train_iter), ' 测试集批数:', len(test_iter))

训练集批数: 235  测试集批数: 40


## 题 1 🔧 填空：nn 三件套（TODO 8.1）

补全优化器。先预测（写你的理解）：

- net 里为什么有 nn.Flatten()？
- 为什么**没有** softmax？（提示：看 loss 是什么）
- init_weights + net.apply 在做什么？

**【你的预测】**

In [2]:
from torch import nn

net = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)
net.apply(init_weights)

loss = nn.CrossEntropyLoss()

# TODO 8.1: 创建 SGD 优化器，学习率 0.1，优化 net 的全部参数
trainer = torch.optim.SGD(net.parameters(), lr = 0.1)

### 读代码：看看 net 和 loss（运行查看）

In [3]:
print('net 结构:', net)
print('Flatten 输出维度:', net[0](torch.zeros(2, 1, 28, 28)).shape)
print('Linear 权重形状:', list(net[1].weight.shape))

net 结构: Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=10, bias=True)
)
Flatten 输出维度: torch.Size([2, 784])
Linear 权重形状: [10, 784]


## 题 2 🔧 补全训练循环（TODO 8.2 ~ 8.4）

与 ex07 从零版对比：这里 loss 已经是标量（CrossEntropyLoss 默认 reduction='mean'），所以直接 .backward()。补全三处 TODO。

**【你的预测】**

In [4]:
def evaluate_accuracy(net, data_iter):          #测评两件套：eval和 no_grad
    """在整个 data_iter 上计算准确率（nn.Module 版）"""
    net.eval()                #切换成评估模式
    metric = 0.0
    n = 0
    with torch.no_grad():     #关闭梯度
        for X, y in data_iter:
            y_hat = net(X)
            metric += float((y_hat.argmax(1) == y).sum())
            n += len(X)
    net.train()
    return metric / n

num_epochs = 10
try:
    for epoch in range(num_epochs):
        train_l_sum = 0.0
        train_acc_sum = 0.0
        n = 0
        for X, y in train_iter:
            y_hat = net(X)
            l = loss(y_hat, y)
            # TODO 8.2: 清零梯度
            trainer.zero_grad()
            # TODO 8.3: 反向传播
            l.backward()
            # TODO 8.4: 更新参数
            trainer.step()
            train_l_sum += float(l) * len(X)        #平均loss * 样本数
            train_acc_sum += float((y_hat.argmax(1) == y).sum())
            n += len(X)           #样本总数++
        test_acc = evaluate_accuracy(net, test_iter)
        print(f'epoch {epoch + 1}, loss {train_l_sum / n:.4f}, train acc {train_acc_sum / n:.3f}, test acc {test_acc:.3f}')
except NotImplementedError as e:
    print(f'⚠ {e}，先完成 TODO 8.2~8.4 再运行')
except NameError as e:
    print(f'⚠ 名字未定义: {e}，先完成 TODO 8.1')

epoch 1, loss 0.7889, train acc 0.746, test acc 0.794
epoch 2, loss 0.5712, train acc 0.813, test acc 0.805
epoch 3, loss 0.5264, train acc 0.825, test acc 0.815
epoch 4, loss 0.5018, train acc 0.832, test acc 0.822
epoch 5, loss 0.4857, train acc 0.837, test acc 0.825
epoch 6, loss 0.4737, train acc 0.841, test acc 0.825
epoch 7, loss 0.4644, train acc 0.844, test acc 0.830
epoch 8, loss 0.4581, train acc 0.845, test acc 0.829
epoch 9, loss 0.4520, train acc 0.846, test acc 0.825
epoch 10, loss 0.4472, train acc 0.848, test acc 0.829


## 题 3 🌱 对比与问答

先写你的回答，再对照答案文件：

1. 从零版（ex07）和简洁版（ex08）的测试准确率大约各是多少？简洁版代码少了多少行？
2. CrossEntropyLoss 内部做了什么，才让 net 里能省掉 softmax？
3. 训练循环里 loss 是标量（mean）还是向量？和 ex07 的「求和成标量」什么关系？

**【你的预测】**

In [ ]:
test_acc = evaluate_accuracy(net, test_iter)
print(f'最终测试准确率: {test_acc:.3f}')

## 小结与面试衔接

- 简洁版 = nn.Flatten + nn.Linear + nn.CrossEntropyLoss + optim.SGD 四个对象，比从零版少十几行
- nn.CrossEntropyLoss 内部 = softmax + 取正确类负对数 + （默认）取平均，所以模型只输出 logits 即可
- 面试高频：CrossEntropyLoss 的输入是「未归一化的 logits」，不要在外面再套 softmax（套了就错了）；为什么它内置 softmax 时还要数值稳定（内部做了 log-sum-exp）
- 训练三步 zero_grad → backward → step 与 ex03 完全一致，是 PyTorch 训练骨架模板